In [1]:
import unicodedata
import numpy as np
import pandas as pd
from tensorflow.keras.utils import Sequence
from tensorflow.keras.layers import Conv2D,Dense,Dropout,Input,LSTM,Embedding,MultiHeadAttention,LayerNormalization
import os
from tensorflow.keras.callbacks import EarlyStopping
import datasets
from datasets import Dataset,DatasetDict
import tensorflow as tf
import re
from tensorflow.keras.preprocessing.sequence import pad_sequences
import ctypes

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

C:\Users\LOQ\AppData\Roaming\Python\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ctypes.windll.kernel32.SetThreadExecutionState(0x80000002)

-2147483648

In [3]:
with open ("D:/project_deeplearning/TEP.en-fa.en",encoding="utf-8") as f:
    en_s=f.read().splitlines()

with open ("D:/project_deeplearning/TEP.en-fa.fa",encoding="utf-8") as f:
    fa_s=f.read().splitlines()


assert len(en_s)==len(fa_s)

df=pd.DataFrame(
    {
        "en":en_s,
        "fa":fa_s
    }
)

df=df.sample(n=60000,random_state=42)
df=df.reset_index(drop=True)

en=df["en"]
fa=df["fa"]
data=pd.DataFrame({"train":Dataset.from_pandas(df)})
print(data["train"][0])

{'en': 'stop her . somebody stop her reading .', 'fa': 'متوقفش كنيد يک نفر نگذاره اون ادامه بده .'}


In [4]:
train=data["train"]
train[:10]

0    {'en': 'stop her . somebody stop her reading ....
1        {'en': 'tetrastichous .', 'fa': 'چهاربيتي .'}
2    {'en': 'thats her stage name . i just said tha...
3    {'en': 'i wanna go . what .', 'fa': 'ميخوام بر...
4    {'en': 'im going to see the dragon warrior .',...
5    {'en': 'just think that i've received them and...
6    {'en': 'and the food was no different from a p...
7    {'en': 'are a couple jerkoffs .', 'fa': '2تا ا...
8    {'en': 'i think i should probably just stay wi...
9                {'en': 'vail .', 'fa': 'بکارخوردن .'}
Name: train, dtype: object

In [5]:
def unicode_to_asci(s):
    return "".join(c for c in unicodedata.normalize("NFC",s) if unicodedata.category(c)!='Mn' )

In [6]:
len(data)

60000

In [7]:
def preprossing(w):
    w=unicode_to_asci(w.lower().strip())
    w = re.sub(r"([.!?])", r" \1 ", w)
    w=re.sub(r'([""])+',"",w)
    w=w.rstrip().strip()
    w = "<start> " + w + " <end>"
    return w

In [8]:
en_sen="Im very happy."
preprossing(en_sen)

'<start> im very happy . <end>'

In [9]:
fa_sen="درود بر تو."
preprossing(fa_sen)

'<start> درود بر تو . <end>'

In [10]:
df["en"]=df["en"].apply(preprossing)
df["fa"]=df["fa"].apply(preprossing)

In [11]:
max_length=35
batch_size=27

trainer=BpeTrainer(vocab_size=5000)

token_en=Tokenizer(BPE(unk_token="<unk>"))
token_fa=Tokenizer(BPE(unk_token="<unk>"))

token_en.add_special_tokens(["<pad>", "<unk>", "<start>", "<end>"])
token_fa.add_special_tokens(["<pad>", "<unk>", "<start>", "<end>"])
token_en.pre_tokenizer=Whitespace()
token_fa.pre_tokenizer=Whitespace()

token_en.train_from_iterator(df["en"].tolist(), trainer)
token_fa.train_from_iterator(df["fa"].tolist(), trainer)

def encode(tokenizer, text):
    return tokenizer.encode(text).ids


In [12]:
en_seq=[encode(token_en, t) for t in df["en"]]
fa_seq=[encode(token_fa, t) for t in df["fa"]]

en_seq=pad_sequences(en_seq,maxlen=max_length,padding="post")
fa_seq=pad_sequences(fa_seq,maxlen=max_length,padding="post")

inputs=fa_seq[:, :-1]
targets=fa_seq[:, 1:]

decoder_seq_lenght=max_length-1
vocab_size_en=token_en.get_vocab_size()
vocab_size_fa=token_fa.get_vocab_size()

In [13]:
latent_dim=256

encoder_inputs=Input(shape=(max_length,),name="encoder_inputs")
encoder_embedding=Embedding(input_dim=vocab_size_en,output_dim=latent_dim,mask_zero=True)(encoder_inputs)
encoder_output,state_h,state_c=LSTM(latent_dim,return_state=True,return_sequences=True,dropout=0.4)(encoder_embedding)



pad_id=token_en.token_to_id("<pad>")
decoder_inputs=Input(shape=(decoder_seq_lenght,),name="decoder_inputs")
decoder_embedding=Embedding(input_dim=vocab_size_fa,output_dim=latent_dim,mask_zero=True,name="decoder_embedding")
decoder_embedd=decoder_embedding(decoder_inputs)
decoder_lstm=LSTM(latent_dim,return_sequences=True,return_state=True,name="decoder_lstm",dropout=0.4)
decoder_outputs,state_h_dec,state_c_dec=decoder_lstm(decoder_embedd,initial_state=[state_h,state_c])
decoder_outputs=Dropout(0.4)(decoder_outputs)
attention_layers=MultiHeadAttention(num_heads=4,key_dim=32,name='multi_head_attention')
attn=attention_layers(query=decoder_outputs,key=encoder_output,value=encoder_output)
x=decoder_outputs+attn
nor=LayerNormalization(name="layer_norm")
x=nor(x)

In [14]:
decoder_dense=Dense(vocab_size_fa,activation="softmax")
decoder_outputs=decoder_dense(x)

In [15]:
model=tf.keras.Model([encoder_inputs,decoder_inputs],decoder_outputs)

In [16]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_inputs      │ (None, 35)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_inputs      │ (None, 34)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 35, 256)   │  1,280,512 │ encoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 35)        │          0 │ encoder_inputs[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_embedding   │ (None, 34, 256)   │  1,280,512 │ decoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ [(None, 35, 256), │    525,312 │ embedding[0][0],  │
│                     │ (None, 256),      │            │ not_equal[0][0]   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, 34, 256), │    525,312 │ decoder_embeddin… │
│                     │ (None, 256),      │            │ lstm[0][1],       │
│                     │ (None, 256)]      │            │ lstm[0][2]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 34, 256)   │          0 │ decoder_lstm[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 34, 256)   │    131,712 │ lstm[0][0],       │
│ (MultiHeadAttentio… │                   │            │ dropout[0][0],    │
│                     │                   │            │ lstm[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 34, 256)   │          0 │ dropout[0][0],    │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_norm          │ (None, 34, 256)   │        512 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 34, 5002)  │  1,285,514 │ layer_norm[0][0]  │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 5,029,386 (19.19 MB)

 Trainable params: 5,029,386 (19.19 MB)

 Non-trainable params: 0 (0.00 B)

In [17]:
n=len(en_seq)
train_end=int(n*0.85)
val_end=int(n * 0.95)

en_train=en_seq[:train_end]
en_val=en_seq[train_end:val_end]
en_test=en_seq[val_end:]

fa_train=fa_seq[:train_end]
fa_val=fa_seq[train_end:val_end]
fa_test=fa_seq[val_end:]

inputs_train=fa_train[:, :-1]
targets_train=fa_train[:, 1:]

inputs_val=fa_val[:, :-1]
targets_val=fa_val[:, 1:]

inputs_test=fa_test[:, :-1]
targets_test=fa_test[:, 1:]

train_ds=tf.data.Dataset.from_tensor_slices(((en_train, inputs_train), targets_train)).shuffle(30000).batch(
    batch_size).prefetch(tf.data.AUTOTUNE)
val_ds=tf.data.Dataset.from_tensor_slices(((en_val, inputs_val), targets_val)).batch(batch_size).prefetch(
    tf.data.AUTOTUNE)
test_ds=tf.data.Dataset.from_tensor_slices(((en_test, inputs_test), targets_test)).batch(batch_size).prefetch(
    tf.data.AUTOTUNE)

In [18]:
loss_ob=tf.keras.losses.SparseCategoricalCrossentropy(reduction="none",from_logits=False)

In [19]:
def msk_loss(y_true,y_pred):

    loss=loss_ob(y_true,y_pred)

    mask=tf.cast(tf.not_equal(y_true,0),dtype=loss.dtype)
    loss=loss*mask
    return tf.reduce_sum(loss)/tf.reduce_sum(mask)

In [20]:
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.005),metrics=[],loss=msk_loss)

In [21]:
early=EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)


In [22]:
history=model.fit(train_ds,epochs=120,validation_data=val_ds,verbose=2,callbacks=[early])

Epoch 1/120
2000/2000 - 370s - 185ms/step - loss: 5.5453 - val_loss: 5.1179
Epoch 2/120
2000/2000 - 290s - 145ms/step - loss: 4.9441 - val_loss: 4.7195
Epoch 3/120
2000/2000 - 645s - 322ms/step - loss: 4.5870 - val_loss: 4.4615
Epoch 4/120
2000/2000 - 633s - 316ms/step - loss: 4.3557 - val_loss: 4.3390
Epoch 5/120
2000/2000 - 612s - 306ms/step - loss: 4.2022 - val_loss: 4.2632
Epoch 6/120
2000/2000 - 556s - 278ms/step - loss: 4.0927 - val_loss: 4.2239
Epoch 7/120
2000/2000 - 449s - 225ms/step - loss: 4.0138 - val_loss: 4.2002
Epoch 8/120
2000/2000 - 158s - 79ms/step - loss: 3.9541 - val_loss: 4.1704
Epoch 9/120
2000/2000 - 158s - 79ms/step - loss: 3.9047 - val_loss: 4.1331
Epoch 10/120
2000/2000 - 158s - 79ms/step - loss: 3.8655 - val_loss: 4.1182
Epoch 11/120
2000/2000 - 334s - 167ms/step - loss: 3.8308 - val_loss: 4.1148
Epoch 12/120
2000/2000 - 550s - 275ms/step - loss: 3.8042 - val_loss: 4.1153
Epoch 13/120
2000/2000 - 332s - 166ms/step - loss: 3.7793 - val_loss: 4.1129
Epoch 14/12

In [24]:

reverse_fa = {idx: token_fa.id_to_token(idx) for idx in range(token_fa.get_vocab_size())}

In [25]:
import pickle
from tensorflow.keras.models import load_model

model=load_model("Translator.keras",custom_objects={"msk_loss":msk_loss})

In [26]:
import pickle
model=load_model("Translator.keras")
pickle.dump(token_fa,open("token_fa.pkl","wb"))
pickle.dump(token_en,open("token_en.pkl","wb"))
pickle.dump(reverse_fa,open("reverse_fa.pkl","wb"))

In [27]:
import pickle

token_fa=pickle.load(open("token_fa.pkl","rb"))
token_en=pickle.load(open("token_en.pkl","rb"))
reverse_fa=pickle.load(open("reverse_fa.pkl","rb"))


In [35]:
encoder_model=tf.keras.Model(encoder_inputs,[encoder_output,state_h,state_c])

In [36]:
decoder_state_input_h=Input(shape=(latent_dim,))
decoder_state_input_c=Input(shape=(latent_dim,))
enc_out_input=Input(shape=(max_length,latent_dim))
decoder_states_inputs=[decoder_state_input_h, decoder_state_input_c]

decoder_emb2=decoder_embedding(decoder_inputs)

decoder_outputs2, state_h2, state_c2 = decoder_lstm(
    decoder_emb2,
    initial_state=decoder_states_inputs
)
attn2=attention_layers(query=decoder_outputs2,key=enc_out_input,value=enc_out_input)
x2=decoder_outputs2+attn2
x2=nor(x2)
decoder_outputs2 = decoder_dense(x2)


decoder_model=tf.keras.Model(
    [decoder_inputs,decoder_state_input_h,decoder_state_input_c,enc_out_input],
    [decoder_outputs2, state_h2, state_c2]
)




In [37]:
encoder_model.save('encoder_model.keras')
decoder_model.save('decoder_model.keras')

In [43]:
def translate(sentence):
    sentence = preprossing(sentence)

    seq=[token_en.encode(sentence).ids]
    seq=pad_sequences(seq, maxlen=max_length, padding="post")

    enc_out, h, c=encoder_model.predict(seq)

    enc_out=tf.convert_to_tensor(enc_out)
    h=tf.convert_to_tensor(h)
    c=tf.convert_to_tensor(c)

    start_token_id=token_fa.token_to_id("<start>")

    target_seq=tf.constant([[start_token_id]])

    stop=False
    decoded = ""

    while not stop:
        output_tokens, h, c =decoder_model([target_seq, h, c, enc_out])

        sampled_token_index=np.argmax(output_tokens[0, -1, :])
        sampled_word=reverse_fa.get(sampled_token_index, "")

        if sampled_word=="<end>" or sampled_word == "" or len(decoded.split())>max_length:
            stop = True
        else:
            decoded += " " +sampled_word

        target_seq=tf.constant([[sampled_token_index]])

    return decoded.strip()

In [44]:
print(translate("I love you"))


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
من دوست دارم


In [45]:
print(translate('you can'))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
تو ميتوني


In [46]:
import sacrebleu
txt="من دوست دارم"
reverse=[" دوستت دارم ","من تورو دوست دارم"]
blu=sacrebleu.sentence_bleu(txt,reverse,tokenize="none")
print(f"score:{blu.score:.2f}")

score:63.00


In [ ]:
print(tf.config.list_physical_devices('GPU'))

In [49]:
import sacrebleu
import re
import random

test_en=df["en"].tolist()[-test_size:]
test_fa=df["fa"].tolist()[-test_size:]

def clean_text(text):
    text=text.replace("<start>", "").replace("<end>", "")
    text=re.sub(r"([.!?،؟])", r" \1", text)
    return re.sub(r"\s+", " ", text).strip()

model_translations=[]
for i, en_sent in enumerate(test_en[:200]):
    translated=translate(en_sent)
    model_translations.append(clean_text(translated))
    if (i+1)%50==0:
        print(f"{i+1}")

references=[[clean_text(ref)] for ref in test_fa[:200]]

bleu=sacrebleu.corpus_bleu(
    model_translations,
    references,
    tokenize='none'
)

print("\n" + "="*50)
print(f"BLEU Score:{bleu.score:.2f}")
print(f"Brevity Penalty:{bleu.bp:.3f}")
print(f"Length Ratio:{bleu.sys_len/bleu.ref_len:.3f}")
print("="*50)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━